# Tutorial 11: Working with Structured Data

In this tutorial, we'll explore how to work with structured data in LangChain and LangGraph applications. We'll use Pydantic for data modeling, create structured inputs and outputs, and leverage the JSON Toolkit for complex data manipulation.

## Setup

First, let's import the necessary libraries and set up our environment:

In [7]:
import os
import json
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, END
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Initialize Groq LLM
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

print("Setup complete.")

Setup complete.


## 1. Introduction to Pydantic for data modeling

Pydantic is a powerful library for data validation and settings management using Python type annotations. Let's start by creating a simple Pydantic model:

In [8]:
class Person(BaseModel):
    name: str
    age: int
    email: Optional[str] = None

# Create a Person instance
alice = Person(name="Alice", age=30, email="alice@example.com")
print(alice)

# Pydantic will raise a validation error if the data doesn't match the model
try:
    invalid_person = Person(name="Bob", age="thirty")
except ValueError as e:
    print(f"Validation error: {e}")

name='Alice' age=30 email='alice@example.com'
Validation error: 1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='thirty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## 2. Creating structured inputs and outputs with Pydantic

Now, let's use Pydantic with LangChain to create structured outputs from our LLM:

In [9]:
class MovieReview(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    rating: float = Field(description="The rating from 0 to 10")
    summary: str = Field(description="A brief summary of the movie's plot")

# .with_structured_output() is the modern replacement for PydanticOutputParser + LLMChain
structured_llm = llm.with_structured_output(MovieReview)

result = structured_llm.invoke("Provide a brief review for the movie The Matrix.")
print(result)

title='The Matrix' year=1999 rating=8.7 summary='In a dystopian future, a computer hacker named Neo discovers that his entire life has been a simulation created by intelligent machines. He is rescued by the resistance leader Morpheus, who reveals the true nature of the world and trains Neo to control the Matrix.'


## 3. Querying JSON Data with an LLM

Python's built-in JSON handling combined with an LLM agent is a clean way to answer questions about structured JSON data.

In [10]:
def generate_json_format_instructions(json_data, parent_key=''):
    """Generate format instructions for a JSON data structure."""
    instructions = {}
    if isinstance(json_data, dict):
        for key, value in json_data.items():
            current_key = f"{parent_key}.{key}" if parent_key else key
            if isinstance(value, dict):
                instructions[key] = {'type': 'object', 'properties': generate_json_format_instructions(value, current_key)}
            elif isinstance(value, list) and value and isinstance(value[0], dict):
                instructions[key] = {'type': 'array', 'items': generate_json_format_instructions(value[0], current_key)}
            elif isinstance(value, list):
                instructions[key] = {'type': 'array', 'items': {'type': type(value[0]).__name__ if value else 'any'}}
            else:
                instructions[key] = {'type': type(value).__name__, 'example': str(value)}
    return instructions

print("JSON format helper defined.")

JSON format helper defined.


In [11]:
from langchain_core.messages import HumanMessage

# JSON dataset
json_data = {
    "movie": [
        {"title": "Inception", "director": "Christopher Nolan", "year": 2010},
        {"title": "The Shawshank Redemption", "director": "Frank Darabont", "year": 1994},
        {"title": "The Godfather", "director": "Francis Ford Coppola", "year": 1972}
    ]
}

schema = generate_json_format_instructions(json_data)
print("Schema:", json.dumps(schema, indent=2))

# Use the LLM to answer questions about the data (modern approach — no deprecated agent needed)
data_str = json.dumps(json_data, indent=2)
response = llm.invoke([
    HumanMessage(content=f"Given this JSON data:\n{data_str}\n\nQuestion: What is the oldest movie in the list?")
])
print("\nAnswer:", response.content)

Schema: {
  "movie": {
    "type": "array",
    "items": {
      "title": {
        "type": "str",
        "example": "Inception"
      },
      "director": {
        "type": "str",
        "example": "Christopher Nolan"
      },
      "year": {
        "type": "int",
        "example": "2010"
      }
    }
  }
}

Answer: To find the oldest movie in the list, we need to find the movie with the lowest 'year' value. We can do this by iterating over the list and comparing the 'year' value of each movie. Here's a Python solution for this problem:

```python
import json

# Define the JSON data
json_data = '''
{
  "movie": [
    {
      "title": "Inception",
      "director": "Christopher Nolan",
      "year": 2010
    },
    {
      "title": "The Shawshank Redemption",
      "director": "Frank Darabont",
      "year": 1994
    },
    {
      "title": "The Godfather",
      "director": "Francis Ford Coppola",
      "year": 1972
    }
  ]
}
'''

# Parse the JSON data
data = json.loads(json_da

## 4. Integrating structured data with LangChain and LangGraph

Now, let's create a more complex example that combines structured data with LangChain and LangGraph. We'll build a movie recommendation system:

In [12]:
class Movie(BaseModel):
    title: str
    genre: str
    year: int
    director: str

class MovieRecommendation(BaseModel):
    recommended_movie: Movie
    reason: str

class MovieDatabase:
    def __init__(self):
        self.movies = [
            Movie(title="The Shawshank Redemption", genre="Drama", year=1994, director="Frank Darabont"),
            Movie(title="The Godfather", genre="Crime", year=1972, director="Francis Ford Coppola"),
            Movie(title="Pulp Fiction", genre="Crime", year=1994, director="Quentin Tarantino"),
            Movie(title="The Dark Knight", genre="Action", year=2008, director="Christopher Nolan"),
            Movie(title="Forrest Gump", genre="Drama", year=1994, director="Robert Zemeckis")
        ]

    def get_movies(self):
        return [m.model_dump() for m in self.movies]

movie_db = MovieDatabase()


@tool
def recommend_movie(preferences: str) -> str:
    """Recommend a movie based on user preferences."""
    movies = movie_db.get_movies()
    structured = llm.with_structured_output(MovieRecommendation)
    result = structured.invoke(
        f"Based on preferences: '{preferences}', recommend one movie from: {json.dumps(movies)}"
    )
    return f"Recommended: {result.recommended_movie.title} ({result.recommended_movie.year}) — {result.reason}"


# create_react_agent replaces the deprecated StructuredChatAgent + AgentExecutor
agent = create_react_agent(model=llm, tools=[recommend_movie])

result = agent.invoke({
    "messages": [HumanMessage(content="I'm in the mood for a classic crime movie. What do you recommend?")]
})
print(result["messages"][-1].content)

The Godfather is widely regarded as one of the greatest films of all time, known for its intricate storyline, memorable characters, and operatic scope.


## Conclusion

In this tutorial, we've explored how to work with structured data in LangChain and LangGraph applications. We've covered:

1. Using Pydantic for data modeling and validation
2. Creating structured inputs and outputs with Pydantic and LangChain
3. Using the JSON Toolkit for complex data manipulation
4. Integrating structured data with LangChain and LangGraph in a movie recommendation system

These techniques allow you to create more robust and type-safe applications, making it easier to work with complex data structures in your AI-powered systems.

## Next Steps

To further improve your skills in working with structured data in LangChain and LangGraph applications, consider the following:

1. Experiment with more complex Pydantic models and nested structures
2. Explore other LangChain tools and agents that work with structured data
3. Implement error handling and fallback strategies for parsing failures
4. Integrate external APIs or databases to work with real-world data
5. Develop a full-fledged application that combines multiple structured data techniques